# loss-item-scalar-extract — ex2: diagnose why .item() raises on per-sample loss + fix with .mean().item()

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `loss-item-scalar-extract`. Running the final beacon cell reports progress against the `PyTorch: loss.item() scalar extract` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: loss.item() scalar extract` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`loss-item-scalar-extract`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "loss-item-scalar-extract"
DD_SUBTOPIC = "PyTorch: loss.item() scalar extract"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `.item()` requires a 0-d (scalar) tensor

Ex1 used `.item()` on a scalar loss. The deepening move is the FAILURE mode: `.item()` on a tensor with `numel() > 1` raises `RuntimeError`.

```python
>>> t.tensor([1.0, 2.0]).item()
RuntimeError: a Tensor with 2 elements cannot be converted to Scalar
```

**The standard fix.** Reduce to a scalar first — `.mean().item()`, `.sum().item()`, `.max().item()`. Each picks a SCALAR aggregate; `.item()` then extracts the Python float.

**Why this matters for logging.** A per-sample loss tensor `(B,)` is what `F.cross_entropy(reduction='none')` returns. Trying to log it directly with `.item()` crashes — you'd want `.mean().item()` (average over the batch) or iterate. The error message is helpful but only if you read it.

### Exercise 2 — diagnose why .item() raises on per-sample loss + fix with .mean().item()

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Analyze
> LO: Analyze the failure mode of `.item()` on a multi-element tensor (per-sample loss) by catching the `RuntimeError` and demonstrate the canonical fix — `.mean().item()` and `.sum().item()` — return scalar Python floats.
> Keywords: item, scalar, RuntimeError, reduction
> ```

**KCs targeted:** `item-requires-zero-d-tensor`, `reduce-before-item`

Implement `ex2_item_failure_and_fix(per_sample_loss)`. The deepening variant of ex1.

Inputs:
- `per_sample_loss`: a 1-D `torch.Tensor` of shape `(B,)` — e.g. what `F.cross_entropy(reduction='none')` would return.

Return a dict with EXACTLY these keys:

- `'numel'`: `int`, `per_sample_loss.numel()`.
- `'item_raised'`: `bool`, `True` if `.item()` on the input raises `RuntimeError`, else `False`. (Catch the exception — do NOT let it propagate.)
- `'item_error_msg'`: `str | None`. If `.item()` raised, the str message of the exception. Else `None`.
- `'mean_scalar'`: `float`, `per_sample_loss.mean().item()`. Always works.
- `'sum_scalar'`: `float`, `per_sample_loss.sum().item()`. Always works.
- `'mean_scalar_type'`: `type`, `type(mean_scalar)` — must be Python `float`, not `torch.Tensor`.

Behavior on a 0-d input:
- If `per_sample_loss.numel() == 1`, `.item()` does NOT raise. Set `'item_raised'=False` and `'item_error_msg'=None`. `mean_scalar` and `sum_scalar` still compute (each will equal the single value).

Constraints:
- Catch ONLY `RuntimeError` from `.item()` — don't catch all exceptions.
- Do not mutate the input tensor.

In [ ]:
def ex2_item_failure_and_fix(per_sample_loss: Tensor) -> dict:
    """Demonstrate .item() failure on non-scalar tensor + the .mean().item() fix."""
    raise NotImplementedError()


def _test_ex2():
    # === Multi-element 1-D loss tensor (B=4): .item() must raise ===
    loss = t.tensor([0.5, 1.0, 2.0, 0.5])
    d = ex2_item_failure_and_fix(loss)
    assert d['numel'] == 4
    assert d['item_raised'] is True, f'item must raise on numel>1, got item_raised={d["item_raised"]}'
    assert isinstance(d['item_error_msg'], str)
    assert 'Scalar' in d['item_error_msg'] or 'scalar' in d['item_error_msg'] or '4' in d['item_error_msg'], (
        f'error message should mention scalar or the element count; got {d["item_error_msg"]!r}'
    )
    # === The .mean().item() and .sum().item() fixes return real Python floats ===
    assert d['mean_scalar_type'] is float, f'mean_scalar must be Python float, got {d["mean_scalar_type"]}'
    import math
    assert math.isclose(d['mean_scalar'], 1.0, rel_tol=1e-6)
    assert math.isclose(d['sum_scalar'], 4.0, rel_tol=1e-6)

    # === 0-d tensor (loss.mean() output already): .item() does NOT raise ===
    scalar = t.tensor(7.5)
    d = ex2_item_failure_and_fix(scalar)
    assert d['numel'] == 1
    assert d['item_raised'] is False, f'item should succeed on 0-d, got item_raised={d["item_raised"]}'
    assert d['item_error_msg'] is None
    assert math.isclose(d['mean_scalar'], 7.5, rel_tol=1e-6)
    assert math.isclose(d['sum_scalar'], 7.5, rel_tol=1e-6)

    # === Single-element 1-D tensor: numel==1, item() also succeeds ===
    single = t.tensor([3.14])
    d = ex2_item_failure_and_fix(single)
    assert d['numel'] == 1
    assert d['item_raised'] is False, 'item must succeed when numel==1, regardless of rank'
    assert d['item_error_msg'] is None
    assert math.isclose(d['mean_scalar'], 3.14, rel_tol=1e-5)

    # === Larger batch ===
    loss = t.arange(10).float()  # [0, 1, ..., 9], sum=45, mean=4.5
    d = ex2_item_failure_and_fix(loss)
    assert d['numel'] == 10
    assert d['item_raised'] is True
    assert math.isclose(d['mean_scalar'], 4.5, rel_tol=1e-6)
    assert math.isclose(d['sum_scalar'], 45.0, rel_tol=1e-6)

    # === Input is not mutated ===
    loss = t.tensor([1.0, 2.0, 3.0])
    loss_clone = loss.clone()
    _ = ex2_item_failure_and_fix(loss)
    assert t.equal(loss, loss_clone), 'input tensor must not be mutated'

    # === Higher-rank multi-element tensor also raises on .item() ===
    loss = t.ones(2, 3)  # numel=6
    d = ex2_item_failure_and_fix(loss)
    assert d['numel'] == 6
    assert d['item_raised'] is True
    assert math.isclose(d['mean_scalar'], 1.0, rel_tol=1e-6)
    assert math.isclose(d['sum_scalar'], 6.0, rel_tol=1e-6)

    # === sum_scalar is Python float, not a 0-d tensor ===
    assert type(d['sum_scalar']) is float, f'sum_scalar must be Python float, got {type(d["sum_scalar"]).__name__}'

    # === All keys ===
    expected_keys = {'numel', 'item_raised', 'item_error_msg', 'mean_scalar',
                     'sum_scalar', 'mean_scalar_type'}
    assert set(d.keys()) == expected_keys, f'keys wrong: {set(d.keys())}'
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def ex2_item_failure_and_fix(per_sample_loss):
    numel = per_sample_loss.numel()
    item_raised = False
    item_error_msg = None
    try:
        _ = per_sample_loss.item()
    except RuntimeError as e:
        item_raised = True
        item_error_msg = str(e)
    mean_scalar = per_sample_loss.mean().item()
    sum_scalar = per_sample_loss.sum().item()
    return {
        'numel': numel,
        'item_raised': item_raised,
        'item_error_msg': item_error_msg,
        'mean_scalar': mean_scalar,
        'sum_scalar': sum_scalar,
        'mean_scalar_type': type(mean_scalar),
    }
```

**`.item()` on numel==1 succeeds REGARDLESS of rank.** A `(1, 1, 1)` tensor with one element calls `.item()` fine. The check inside PyTorch is `numel() == 1`, not `ndim == 0`. This matters when a batch happens to be size 1 — your `.item()` calls won't crash, but they will crash the moment batch grows.

**Catch `RuntimeError`, not `Exception`.** PyTorch raises `RuntimeError` specifically. Catching a broader exception class would swallow unrelated bugs (e.g. an `AttributeError` from a typo). Narrow except clauses are how you keep error handling focused.

**`type(x) is float` for the Python-float check.** `isinstance(x, float)` is also fine here — both `True` and `False` for `float` and not `Tensor`. The distinction matters more when a subclass is involved; here either works.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()